<h1> Parsing </h1>

In [ ]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning #txt files
from unstructured.partition.pdf import partition_pdf #pdf files
from unstructured.partition.docx import partition_docx #word files

from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking
from unstructured.chunking.basic import chunk_elements

from pathlib import Path
from langchain_core.documents import Document

In [2]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)

    elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos


<hr>

<h1> Retrieval</h1>

<h3> Sparse Retrieval </h3>

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10) #@K aqui definido

<hr>

<h3> Dense Retrieval </h3>

<h5> Embeddings & Indexing </h5>

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção


emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

In [5]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

<h3> Reciprocal Rank Fusion </h3>

$$
RRF (d) = \sum _{i = 1}^{n} \frac {1} {k + pos_i(d)} 
$$

In [6]:
from eval.HitRate import Reciprocal_Rank_Fusion

### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query) #Procura Lexical
d_retrieval = dense_retrieval.invoke (query) #Procura Vetorial

#display (s_retrieval)
#display (d_retrieval)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

#for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
#    print (ordem)
#    print (doc_name)
#    print (chunk_id)
#    print (chunk_text)
#    print (score)

#display (len(teste))
#display (teste)

<h2> Eval Retrieval </h2>

In [7]:
#Dataset
import json

with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset.json", "r", encoding = "utf-8") as f:
    dataset = json.load(f)

from eval.HitRate import hitrate_k_sparse_retrieval, hitrate_k_dense_retrieval, hitrate_k_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval


<h3> HitRate@5 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 5)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 5
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> HitRate@10 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> HitRate@15 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 15)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> MRR </h3>

In [11]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = mrr_sparse_retrieval (sparse_retrieval_1, dataset)
y = mrr_dense_retrieval (dense_retrieval_1, dataset)
z = mrr_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

{'Mean Reciprocal Rank Chunk Sparse Retrieval': {0.38395492311507934},
 'Mean Reciprocal Rank Docs Sparse Retrieval': {0.9459635416666666}}

{'Mean Reciprocal Rank Chunks Dense Retrieval': {0.2799556671626985},
 'Mean Reciprocal Rank Docs Dense Retrieval': {0.8357406374007936}}

{'Mean Reciprocal Rank Chunks Hybrid Retrieval': {0.3886640247135422},
 'Mean Reciprocal Rank Docs Hybrid Retrieval': {0.9483506944444444}}

<hr>

<h6> código solto </h6>

In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 20
    }
)

#x = dense_retrieval.invoke ("O que é a corrente dinâmica estipulada (Idyn)?")

In [ ]:
### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query)
d_retrieval = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
    print (ordem)
    print (doc_name)
    print (chunk_id)
    print (chunk_text)
    print (score)

#display (teste)


<h1> Reranking </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path)
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

In [ ]:
#features = [query] * len (teste), [t[0] for t in teste]
'''
[('Crianças_e_jovens_(9-17)_e_Inteligência_Artificial_Generativa_em_Portugal.pdf',
  222,
  'A Rede EU KIDS ONLINE e o estudo por detrás deste relatório\n\nA rede EU Kids Online, uma rede colaborativa e independente, reúne investigadores de 26 países. Sob uma perspetiva multidisciplinar e multimetodológica, investigam oportunidades, riscos e segurança de crianças e jovens na internet. Ver www.eukidsonline.net',
  0.032018442622950824),
'''
query_reranker = [query] * len (teste)
chunks = [t[2] for t in teste]
docs_names = [t[0] for t in teste]
chunks_ids = [t[1] for t in teste]

pares = list(zip(query_reranker, chunks)) # [query, chunks]
#print (pares)

inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder_model.eval()
with torch.no_grad():
    logits = cross_encoder_model (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
rerank = sorted(zip(docs_names, chunks_ids, chunks, logits.tolist()), key = lambda x: x[3], reverse = True) #x[3] define que arg é usado pelo reverse

display (rerank)

In [ ]:
final = [(docs, chunks) for docs, chunks, chunks_content, scores in rerank [:5]]

display (final)

In [ ]:
from eval.HitRate import hitrate_k_reranker

h = hitrate_k_reranker (sparse_retrieval, dense_retrieval, dataset)

In [8]:
from eval.MeanReciprocalRank import mrr_ranker_sparse_system

f = mrr_ranker_sparse_system (sparse_retrieval, dataset)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1709.38it/s]


In [9]:
print (f)

{'Mean Reciprocal Rank Chunks Rerank': {0.4693359374999999}, 'Mean Reciprocal Rank Docs Rerank': {0.9698567708333334}}


<h3> LLM </h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


In [ ]:
prompt = "Em que consiste o relatório da rede EU Kids Online ?"


chat_template = f"""
<|im_start|>
És um assistente especializado em responder exclusivamente com base no contexto fornecido.

Regras:
1. Utiliza apenas a informação do Contexto.
2. Não inventes informação.
3. Não uses conhecimento externo.
4. Se a resposta não existir no contexto, responde exatamente:
   "Informação não encontrada na base de dados."
5. Responde de forma breve e direta.
6. Formata a resposta de forma clara.

Contexto:
{feed_llm}
<|im_end|>

<|im_start>
{prompt}
<|im_end|>

<|im_start|>
"""

model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens = 512)

input_length = model_inputs["input_ids"].shape[1]
response_tokens = generated_ids[0][input_length:]

print(tokenizer2.decode(response_tokens, skip_special_tokens = True))

<hr>

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    sparse_retr = sparse_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Sparse:", mrr)

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    dense_retr = dense_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Dense:", mrr)

In [ ]:
rrf_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    rankings = [
        dense_retrieval.invoke(query),
        sparse_retrieval.invoke(query)
    ]

    rrf_ranked = Reciprocal_Rank_Fusion(rankings)

    rr = 0

    for rank, (doc_content, score, doc_id) in enumerate(rrf_ranked, start=1):
        if doc_id == gold_doc:
            rr = 1 / rank
            break

    rrf_scores.append(rr)

mrr_rrf = sum(rrf_scores) / len(rrf_scores)

print("MRR RRF:", mrr_rrf)